### Import libraries

In [1]:
import os
import json
from pathlib import Path

LOCAL_DATA_ROOT = Path(os.getcwd()).resolve().parent / "data"

import torch
import numpy as np
from datasets import load_dataset
from torch.utils.data import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


/home/doterog/repos/personal_projects/continuous-flow-asr/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Set HPs

In [5]:
REPO_ID = "openslr/librispeech_asr"
CONFIG = "clean"
SPLIT = "train.100"
DEFAULT_MODEL = "openai/whisper-large-v3"

### Load dataset

In [ ]:
audio_feature_dir = (
    LOCAL_DATA_ROOT /
    "features" /
    DEFAULT_MODEL.split("/")[-1] /
    CONFIG /
    SPLIT
)

class AudioFeatureDataset(Dataset):
    """Utterance-level view over the sharded (frames, d_model) feature files.

    The split is ~44 GB on disk, so the .npy shards stay memory-mapped and only the
    requested slice is copied into RAM; the OS page cache keeps hot shards resident.
    The .json indexes are small, so they are read once up front and flattened into a
    single utterance table (shards are not all the same size, so idx can't be split
    arithmetically).
    """

    def __init__(self, audio_feature_dir: Path):
        self.index_paths = sorted(audio_feature_dir.glob("*.json"))
        assert self.index_paths, f"no feature shards under {audio_feature_dir}"

        self.utts = []  # (shard_idx, offset, length, text)
        for shard_idx, index_path in enumerate(self.index_paths):
            index = json.loads(index_path.read_text())
            self.utts += [
                (shard_idx, u["offset"], u["length"], u["text"]) for u in index["utts"]
            ]
        self.d_model = index["d_model"]
        self._shards = {}  # opened lazily so each DataLoader worker gets its own mmap

    def __len__(self):
        return len(self.utts)

    def _feats(self, shard_idx: int) -> np.ndarray:
        if shard_idx not in self._shards:
            path = self.index_paths[shard_idx].with_suffix(".npy")
            self._shards[shard_idx] = np.load(path, mmap_mode="r")
        return self._shards[shard_idx]

    def __getitem__(self, idx):
        shard_idx, offset, length, text = self.utts[idx]
        feats = self._feats(shard_idx)[offset:offset + length]
        return {
            # copy off the read-only mmap so the tensor owns its memory
            "audio": torch.from_numpy(np.array(feats)),
            "text": text,
        }


audio_ds = AudioFeatureDataset(audio_feature_dir)


28539 utterances, d_model=1280


(torch.Size([727, 1280]),
 'CHAPTER SIXTEEN I MIGHT HAVE TOLD YOU OF THE BEGINNING OF TH')